# Forecast ABS TTE — CatBoost por Componente

**CatBoost (por componente) | Moirai | Chronos 2 | TimesFM 2.5 (sobre ABS_Total)**

Fuente: `meli-people.SILVER_PE_SHIPPING.FORECASTING_ABS_FINANCIAL_PLANNING_TTE` (MELI + EXTERNO combinados, sin filtro de `TIPO_CONTRATO`)

Contiene:
1. **Importacion de datos** — Carga directa con `bigquery.Client` + pivot de `TIPO_ABS_GESTIONABLE`
2. **Feature Engineering** — Features temporales, lags, medias moviles, tendencia
3. **CatBoost por componente** — Un modelo in-sample por `Operational_name` y variable (Dotacion, Falta_Inj, Atestados_Medicos, Rest_Abs_Gestionable)
4. **Benchmark TSS** — Moirai, Chronos-2 y TimesFM 2.5 sobre el agregado `ABS_Total` (walk-forward validation + produccion)
5. **Consolidacion, dashboard y exportacion a BigQuery**

---
**Variables pivoteadas desde `TIPO_ABS_GESTIONABLE`:**
- Dotacion (`a.Dotacion`)
- Falta_Inj (`b.Falta_Inj`)
- Atestados_Medicos (`c.Atestados_Medicos`)
- Rest_Abs_Gestionable (`d.rest_abs_gestionable`)

**Metricas derivadas:**
- ABS_Falta_Inj, ABS_Atestados_Medicos, ABS_Rest_Abs_Gestionable = componente / Dotacion
- Ausencia_Total = suma de los 3 componentes de ausencia
- ABS_Total = Ausencia_Total / Dotacion — comparado entre CatBoost, Moirai, Chronos-2 y TimesFM 2.5

---
### Celda 1 — Imports y Configuracion Global

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from dateutil.relativedelta import relativedelta
from google.cloud import bigquery
import json, os, uuid, time, requests, warnings

warnings.filterwarnings('ignore')

# =============================================================================
# PARAMETROS DE CONFIGURACION
# =============================================================================

PROJECT_ID = "meli-people"
BQ_TABLE_SOURCE = "meli-people.DW_PE_SHIPPING.FORECASTING_ABS_FINANCIAL_PLANNING_TTE"
BQ_TABLE_DESTINATION = "meli-people.SILVER_PE_SHIPPING.FORECASTING_ABS_FINANCIAL_PLANNING_TTE_FORECAST"

COLUMNAS_FORECAST = ['Dotacion', 'Falta_Inj', 'Atestados_Medicos', 'Rest_Abs_Gestionable']
COLUMNAS_AUSENCIA = ['Falta_Inj', 'Atestados_Medicos', 'Rest_Abs_Gestionable']

OUTPUT_DIR = os.path.join(os.getcwd(), 'forecast_results_ABS_TTE_catboost')
os.makedirs(OUTPUT_DIR, exist_ok=True)
RAW_DIR = os.path.join(OUTPUT_DIR, 'raw_tss')
CSV_DIR = os.path.join(OUTPUT_DIR, 'csv_tss_input')
for d in [RAW_DIR, CSV_DIR]:
    os.makedirs(d, exist_ok=True)

# --- Fechas automaticas ---
FECHA_ACTUAL = pd.Timestamp.now()
MES_ACTUAL = FECHA_ACTUAL.replace(day=1)
MES_ANTERIOR = (MES_ACTUAL - relativedelta(months=1))

FECHA_INICIO_HISTORICO = pd.Timestamp('2024-01-01')
FECHA_LIMITE_HISTORICO = MES_ANTERIOR
FECHA_FIN_FORECAST = pd.Timestamp('2027-12-01')

FECHAS_FORECAST = list(pd.date_range(start=MES_ACTUAL, periods=18, freq='MS'))  # 18 meses fijos: Jul 2026 -> Dic 2027
N_MONTHS_FORECAST = len(FECHAS_FORECAST)

# --- Validacion walk-forward (dinamica, no hardcodeada) ---
VAL_HORIZON = 6
_val_cutoff_month = (MES_ANTERIOR - relativedelta(months=VAL_HORIZON)).replace(day=1)
VAL_CUTOFF = _val_cutoff_month + pd.offsets.MonthEnd(0)
VAL_TEST_INI = _val_cutoff_month + relativedelta(months=1)

VAR1, VAR2, METRIC = 'Ausencia_Total', 'Dotacion', 'ABS'

print("=" * 70)
print("FORECAST ABS TTE — PIVOTEADO (MELI+EXTERNO) + BENCHMARK 6 MODELOS (TSS por componente)")
print("=" * 70)
print(f"Fecha actual        : {FECHA_ACTUAL.strftime('%Y-%m-%d')}")
print(f"Historico           : {FECHA_INICIO_HISTORICO.strftime('%Y-%m')} -> {FECHA_LIMITE_HISTORICO.strftime('%Y-%m')}")
print(f"Forecast ({N_MONTHS_FORECAST} meses)  : {MES_ACTUAL.strftime('%Y-%m')} -> {FECHA_FIN_FORECAST.strftime('%Y-%m')}")
print(f"Validacion walk-fwd : cutoff={VAL_CUTOFF.strftime('%Y-%m')}, horizon={VAL_HORIZON} meses")
print(f"Variables componente: {', '.join(COLUMNAS_FORECAST)}")
print(f"Output dir          : {OUTPUT_DIR}")
print("=" * 70)


FORECAST ABS TTE — PIVOTEADO (MELI+EXTERNO) + BENCHMARK 6 MODELOS (TSS por componente)
Fecha actual        : 2026-07-17
Historico           : 2024-01 -> 2026-06
Forecast (18 meses)  : 2026-07 -> 2027-12
Validacion walk-fwd : cutoff=2025-12, horizon=6 meses
Variables componente: Dotacion, Falta_Inj, Atestados_Medicos, Rest_Abs_Gestionable
Output dir          : c:\Users\jhocontreras\Desktop\TTE_TIME_SERIES\ABS GEST FINANCIAL\forecast_results_ABS_TTE_catboost


---
### Celda 2 — Importacion de Datos desde BigQuery (Pivot MELI + EXTERNO)

In [2]:
# =============================================================================
# CARGA + PIVOT DIRECTO DESDE BIGQUERY (cliente directo, MELI + EXTERNO)
# =============================================================================

client = bigquery.Client(project=PROJECT_ID)

query = f"""
WITH base AS (
  SELECT ANO, MES, Operational_name, SITE, Tipo_OPS,
         TIPO_ABS_GESTIONABLE,
         (Dotacion_programada + Ausentismo_gestionable) AS VALOR
  FROM `{BQ_TABLE_SOURCE}`
),
agg AS (
  SELECT ANO, MES, Operational_name,
         ANY_VALUE(SITE) AS SITE, ANY_VALUE(Tipo_OPS) AS Tipo_OPS,
         TIPO_ABS_GESTIONABLE, SUM(VALOR) AS VALOR
  FROM base
  GROUP BY ANO, MES, Operational_name, TIPO_ABS_GESTIONABLE
),
pivoted AS (
  SELECT * FROM agg
  PIVOT (
    SUM(VALOR)
    FOR TIPO_ABS_GESTIONABLE IN (
      'a.Dotacion' AS Dotacion,
      'b.Falta_Inj' AS Falta_Inj,
      'c.Atestados_Medicos' AS Atestados_Medicos,
      'd.rest_abs_gestionable' AS Rest_Abs_Gestionable
    )
  )
)
SELECT ANO, MES, Operational_name, SITE, Tipo_OPS,
  COALESCE(Dotacion, 0) AS Dotacion,
  COALESCE(Falta_Inj, 0) AS Falta_Inj,
  COALESCE(Atestados_Medicos, 0) AS Atestados_Medicos,
  COALESCE(Rest_Abs_Gestionable, 0) AS Rest_Abs_Gestionable
FROM pivoted
ORDER BY ANO, MES, Operational_name
"""

print("Cargando y pivoteando datos desde BigQuery (cliente directo)...")
df_raw = client.query(query).to_dataframe()
print(f"✅ {len(df_raw):,} registros cargados")

df_raw['fecha_mes'] = pd.to_datetime(df_raw['ANO'].astype(str) + '-' + df_raw['MES'].astype(str).str.zfill(2) + '-01')

df_raw = df_raw[
    (df_raw['fecha_mes'] >= FECHA_INICIO_HISTORICO) &
    (df_raw['fecha_mes'] <= FECHA_LIMITE_HISTORICO)
].copy()

for col in COLUMNAS_FORECAST:
    df_raw[col] = df_raw[col].astype('float64')

df_raw = df_raw.sort_values(['Operational_name', 'fecha_mes']).reset_index(drop=True)

# --- Ausencia_Total / ABS_Total historico ---
df_raw['Ausencia_Total'] = df_raw[COLUMNAS_AUSENCIA].sum(axis=1)
df_raw['ABS_Total'] = np.where(df_raw['Dotacion'] > 0, (df_raw['Ausencia_Total'] / df_raw['Dotacion']).round(4), 0)

# --- Filtrar sitios con minimo 6 meses de historia ---
meses_x_sitio = df_raw.groupby('Operational_name')['fecha_mes'].count()
SITIOS = sorted(meses_x_sitio[meses_x_sitio >= 6].index.tolist())
df_data = df_raw[df_raw['Operational_name'].isin(SITIOS)].copy()

print(f"Registros en rango : {len(df_data):,}")
print(f"Sitios (>=6 meses) : {len(SITIOS)}")
print(f"Periodo            : {df_data['fecha_mes'].min().strftime('%Y-%m')} -> {df_data['fecha_mes'].max().strftime('%Y-%m')}")
df_data.head()


Cargando y pivoteando datos desde BigQuery (cliente directo)...
✅ 1,506 registros cargados
Registros en rango : 1,374
Sitios (>=6 meses) : 64
Periodo            : 2024-01 -> 2026-06


,ANO,MES,Operational_name,SITE,Tipo_OPS,Dotacion,Falta_Inj,Atestados_Medicos,Rest_Abs_Gestionable,fecha_mes,Ausencia_Total,ABS_Total
0,2025,8,AH - GUARULHOS SSP19,AH - GUARULHOS SSP19,TTE,52.0,5.0,2.0,0.0,2025-08-01,7.0,0.1346
1,2025,9,AH - GUARULHOS SSP19,AH - GUARULHOS SSP19,TTE,157.0,0.0,0.0,0.0,2025-09-01,0.0,0.0000
2,2025,10,AH - GUARULHOS SSP19,AH - GUARULHOS SSP19,TTE,1.0,0.0,0.0,0.0,2025-10-01,0.0,0.0000
3,2025,11,AH - GUARULHOS SSP19,AH - GUARULHOS SSP19,TTE,19.0,0.0,0.0,0.0,2025-11-01,0.0,0.0000
4,2025,12,AH - GUARULHOS SSP19,AH - GUARULHOS SSP19,TTE,45.0,0.0,0.0,0.0,2025-12-01,0.0,0.0000


---
### Celda 3 — Feature Engineering (Temporales, Lags, Medias Moviles, Tendencia)

In [3]:
# =============================================================================
# FEATURE ENGINEERING
# =============================================================================

def crear_features_temporales(df):
    df = df.copy()
    df['mes'] = df['fecha_mes'].dt.month
    df['año'] = df['fecha_mes'].dt.year
    df['trimestre'] = df['fecha_mes'].dt.quarter
    df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
    df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)
    fecha_min = df['fecha_mes'].min()
    df['num_mes'] = ((df['fecha_mes'].dt.year - fecha_min.year) * 12 +
                     (df['fecha_mes'].dt.month - fecha_min.month))
    return df


def crear_features_con_lags(df, target_col, lags=[1, 2, 3, 6]):
    df = df.copy()
    df = crear_features_temporales(df)

    if df[target_col].dtype in ['Int64', 'Int32', 'Int16', 'Int8', 'UInt64', 'UInt32', 'UInt16', 'UInt8']:
        df[target_col] = df[target_col].astype('float64')

    for lag in lags:
        if len(df) > lag:
            df[f'{target_col}_lag_{lag}'] = df[target_col].shift(lag).astype('float64')

    if len(df) >= 3:
        df[f'{target_col}_ma_3'] = df[target_col].rolling(window=3, min_periods=1).mean().astype('float64')
    if len(df) >= 6:
        df[f'{target_col}_ma_6'] = df[target_col].rolling(window=6, min_periods=1).mean().astype('float64')

    df[f'{target_col}_tendencia'] = df[target_col].diff().astype('float64')

    return df


def calcular_mape(y_real, y_pred):
    y_real, y_pred = np.array(y_real, dtype=float), np.array(y_pred, dtype=float)
    mask = y_real > 0
    if mask.sum() == 0:
        return 0.0
    return min(np.mean(np.abs((y_real[mask] - y_pred[mask]) / y_real[mask])) * 100, 200.0)


def calc_metrics_detailed(y_real, y_pred, name=""):
    """Calcula metricas completas (MAPE, WMAPE, SMAPE, Bias, R2) para un par real/predicho."""
    mask = (y_real != 0) & (~np.isnan(y_real)) & (~np.isnan(y_pred))
    y_r, y_p = y_real[mask], y_pred[mask]
    n = len(y_r)
    if n == 0:
        return {}

    errors = y_p - y_r
    abs_errors = np.abs(errors)

    mae = abs_errors.mean()
    rmse = np.sqrt((errors ** 2).mean())
    me = errors.mean()

    mape = (abs_errors / np.abs(y_r) * 100).mean()
    wmape = abs_errors.sum() / np.abs(y_r).sum() * 100
    smape = (2 * abs_errors / (np.abs(y_r) + np.abs(y_p)) * 100).mean()
    mae_pct = mae / np.abs(y_r).mean() * 100

    ss_res = (errors ** 2).sum()
    ss_tot = ((y_r - y_r.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0

    return {
        'n': n, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2),
        'MAPE': round(mape, 2), 'WMAPE': round(wmape, 2),
        'SMAPE': round(smape, 2), 'MAE%': round(mae_pct, 2),
        'Bias(ME)': round(me, 2), 'R2': round(r2, 4)
    }

print("Feature engineering definido (crear_features_temporales, crear_features_con_lags, calcular_mape, calc_metrics_detailed)")


Feature engineering definido (crear_features_temporales, crear_features_con_lags, calcular_mape, calc_metrics_detailed)


---
### Celda 4 — Entrenamiento CatBoost por Sitio y Componente (In-Sample)

In [4]:
# =============================================================================
# MODELOS ML POR SITIO: CatBoost · LightGBM · ElasticNet  (in-sample)
# =============================================================================

class _MLTTEForecaster:
    # Base forecaster: entrena un modelo ML por sitio y variable usando lags/rolling.
    def __init__(self, target_col, name="Model"):
        self.target_col = target_col
        self.name = name
        self.models = {}
        self.metrics = {}

    def _get_feature_cols(self):
        return ['mes', 'año', 'trimestre', 'mes_sin', 'mes_cos', 'num_mes',
                f'{self.target_col}_lag_1', f'{self.target_col}_lag_2',
                f'{self.target_col}_lag_3', f'{self.target_col}_lag_6',
                f'{self.target_col}_ma_3', f'{self.target_col}_ma_6',
                f'{self.target_col}_tendencia']

    def _build_model(self, n_samples):
        raise NotImplementedError

    def fit(self, df, sites):
        print(f"\nEntrenando {self.__class__.__name__} para {self.name}...")
        feature_cols = self._get_feature_cols()
        for site in sites:
            site_data = df[df['Operational_name'] == site].copy().sort_values('fecha_mes')
            site_data = crear_features_con_lags(site_data, self.target_col)
            available_features = [f for f in feature_cols if f in site_data.columns]
            site_data = site_data.dropna(subset=available_features + [self.target_col])
            if len(site_data) < 6:
                continue
            y = site_data[self.target_col].values
            if np.std(y) == 0 or len(np.unique(y)) == 1:
                continue
            try:
                X = site_data[available_features].values.astype('float64')
                X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
                model = self._build_model(len(site_data))
                model.fit(X, y)
                self.models[site] = {
                    'model': model,
                    'features': available_features,
                    'last_data': site_data.tail(6).copy()
                }
                y_pred = np.maximum(model.predict(X), 0)
                self.metrics[site] = {
                    'mae': mean_absolute_error(y, y_pred),
                    'rmse': np.sqrt(mean_squared_error(y, y_pred)),
                    'r2': r2_score(y, y_pred) if len(y) > 1 else 0,
                    'n_samples': len(site_data)
                }
            except Exception as e:
                print(f'      ERROR fit {site}/{self.target_col}: {e}')
        print(f"   {len(self.models)} modelos entrenados de {len(sites)} sitios")
        return self

    def predict_future(self, future_dates):
        predictions = []
        for site, model_info in self.models.items():
            model, features = model_info['model'], model_info['features']
            last_data = model_info['last_data'].copy()
            for future_date in future_dates:
                new_row = pd.DataFrame({'fecha_mes': [future_date], 'Operational_name': [site], self.target_col: [np.nan]})
                temp_df = pd.concat([last_data, new_row], ignore_index=True).sort_values('fecha_mes')
                temp_df = crear_features_con_lags(temp_df, self.target_col)
                future_row = temp_df[temp_df['fecha_mes'] == future_date].copy()
                if len(future_row) > 0:
                    for col in features:
                        if col not in future_row.columns: future_row[col] = 0
                        else: future_row[col] = future_row[col].fillna(0)
                    try:
                        X_pred = future_row[features].values
                        pred = max(0, float(model.predict(X_pred)[0]))
                        predictions.append({'fecha_mes': future_date, 'Operational_name': site, f'{self.target_col}_forecast': pred})
                        last_data.loc[last_data.index.max() + 1] = {'fecha_mes': future_date, 'Operational_name': site, self.target_col: pred}
                        last_data = last_data.tail(6)
                    except Exception:
                        continue
        return pd.DataFrame(predictions)

    def predict_historical(self, df):
        # Predicciones in-sample para calcular MAPE
        predictions = []
        for site, model_info in self.models.items():
            site_data = df[df['Operational_name'] == site].copy().sort_values('fecha_mes')
            if len(site_data) == 0: continue
            site_data = crear_features_con_lags(site_data, self.target_col)
            features = model_info['features']
            for col in features:
                if col in site_data.columns:
                    if 'lag' in col:
                        site_data[col] = site_data[col].bfill().ffill().fillna(site_data[self.target_col].iloc[0] if len(site_data) > 0 else 0)
                    elif 'ma' in col:
                        mean_val = site_data[self.target_col].mean()
                        site_data[col] = site_data[col].fillna(mean_val if pd.notna(mean_val) else 0)
                    else:
                        site_data[col] = site_data[col].fillna(0)
                else:
                    site_data[col] = 0
            try:
                X_pred = site_data[features].values.astype('float64')
                X_pred = np.nan_to_num(X_pred, nan=0.0, posinf=0.0, neginf=0.0)
                y_pred = model_info['model'].predict(X_pred)
                for idx, (_, row) in enumerate(site_data.iterrows()):
                    predictions.append({'fecha_mes': row['fecha_mes'], 'Operational_name': site, f'{self.target_col}_pred': max(0, float(y_pred[idx]))})
            except Exception:
                continue
        return pd.DataFrame(predictions)

    def get_metrics_df(self):
        return pd.DataFrame([{'Operational_name': site, 'variable': self.target_col,
                               'MAE': m['mae'], 'RMSE': m['rmse'], 'R2': m['r2'], 'n_samples': m['n_samples']}
                              for site, m in self.metrics.items()])


class CatBoostTTEForecaster(_MLTTEForecaster):
    def _build_model(self, n_samples):
        return CatBoostRegressor(
            iterations=100 if n_samples >= 15 else 50,
            depth=4, learning_rate=0.08, l2_leaf_reg=1.0, random_seed=42, verbose=False)


# -- Entrenamiento CatBoost --------------------------------------------------
print('=' * 70)
print('ENTRENAMIENTO — CatBoost (in-sample, ABS TTE)')
print('=' * 70)
print(f'Periodo: {FECHA_INICIO_HISTORICO.strftime("%Y-%m")} a {FECHA_LIMITE_HISTORICO.strftime("%Y-%m")}')
print(f'Total sitios: {len(SITIOS)}')

models      = {}
all_metrics = []

for col in COLUMNAS_FORECAST:
    m_cb = CatBoostTTEForecaster(target_col=col, name=col)
    m_cb.fit(df_data, SITIOS)
    models[col] = m_cb
    all_metrics.append(m_cb.get_metrics_df().assign(modelo='CatBoost'))

df_all_metrics = pd.concat(all_metrics, ignore_index=True)

print(chr(10) + '=' * 70)
print('METRICAS DE ENTRENAMIENTO (in-sample, promedio)')
print('=' * 70)
dm = df_all_metrics[df_all_metrics['modelo'] == 'CatBoost']
if len(dm) > 0:
    print(f'CatBoost:  MAE={dm["MAE"].mean():.2f}  RMSE={dm["RMSE"].mean():.2f}  R2={dm["R2"].mean():.4f}  modelos={len(dm)}')


ENTRENAMIENTO — CatBoost (in-sample, ABS TTE)
Periodo: 2024-01 a 2026-06
Total sitios: 64

Entrenando CatBoostTTEForecaster para Dotacion...
   50 modelos entrenados de 64 sitios

Entrenando CatBoostTTEForecaster para Falta_Inj...
   50 modelos entrenados de 64 sitios

Entrenando CatBoostTTEForecaster para Atestados_Medicos...
   50 modelos entrenados de 64 sitios

Entrenando CatBoostTTEForecaster para Rest_Abs_Gestionable...
   50 modelos entrenados de 64 sitios

METRICAS DE ENTRENAMIENTO (in-sample, promedio)
CatBoost:  MAE=17.62  RMSE=22.02  R2=0.9938  modelos=200


---
### Celda 5 — Diagnostico de Performance In-Sample (MAPE, WMAPE, SMAPE, Bias, R2)

In [5]:
# =============================================================================
# DIAGNOSTICO DE PERFORMANCE (in-sample) — CatBoost / LightGBM / ElasticNet
# =============================================================================

print("=" * 90)
print("DIAGNOSTICO DE PERFORMANCE - in-sample  (CatBoost · LightGBM · ElasticNet)")
print("=" * 90)

all_perf = []
hist_preds      = {}   # CatBoost

print("\nCalculando metricas in-sample...\n")
for col in COLUMNAS_FORECAST:
    for tag, mdls, hpreds in [('CB', models, hist_preds)]:
        df_hp = mdls[col].predict_historical(df_data)
        if len(df_hp) == 0:
            continue
        hpreds[col] = df_hp
        if tag == 'CB':
            merged = df_data[['fecha_mes', 'Operational_name', col]].merge(df_hp, on=['fecha_mes', 'Operational_name'], how='inner')
            for site in merged['Operational_name'].unique():
                ds = merged[merged['Operational_name'] == site]
                m = calc_metrics_detailed(ds[col].values, ds[f'{col}_pred'].values)
                if m:
                    m['Variable'] = col
                    m['Operational_name'] = site
                    all_perf.append(m)

df_perf = pd.DataFrame(all_perf)

print(f"{'Variable':<22} {'MAPE%':>7} {'WMAPE%':>8} {'SMAPE%':>8} {'MAE%':>7} {'Bias(ME)':>10} {'R2':>7} {'n_sitios':>9}")
print("-" * 85)
for col in COLUMNAS_FORECAST:
    vc = df_perf[df_perf['Variable'] == col]
    if len(vc) == 0:
        continue
    print(f"{col:<22} {vc['MAPE'].mean():>7.2f} {vc['WMAPE'].mean():>8.2f} {vc['SMAPE'].mean():>8.2f} "
          f"{vc['MAE%'].mean():>7.2f} {vc['Bias(ME)'].mean():>10.1f} {vc['R2'].mean():>7.4f} {len(vc):>9}")

print("\n" + "=" * 90)
print("SEMAFORO DE SALUD POR VARIABLE (CatBoost WMAPE)")
print("=" * 90)
for col in COLUMNAS_FORECAST:
    vc = df_perf[df_perf['Variable'] == col]
    if len(vc) == 0:
        continue
    wmape = vc['WMAPE'].mean()
    bias  = vc['Bias(ME)'].mean()
    emoji = '\U0001f7e2' if wmape < 5 else ('\U0001f7e1' if wmape < 10 else '\U0001f534')
    print(f"  {emoji} {col:<22} WMAPE={wmape:.1f}%  Bias={bias:+.1f} ({'sobreestima' if bias > 0 else 'subestima'})")

# --- ABS_Total comparacion in-sample por modelo ---
print("\n" + "=" * 90)
print("PERFORMANCE ABS_TOTAL — CatBoost vs LightGBM vs ElasticNet (in-sample)")
print("=" * 90)

def _abs_total_insample(hp_dict, label):
    df_m = df_data[['fecha_mes', 'Operational_name', 'Dotacion'] + COLUMNAS_AUSENCIA].copy()
    for col in COLUMNAS_FORECAST:
        if col in hp_dict:
            df_m = df_m.merge(hp_dict[col], on=['fecha_mes', 'Operational_name'], how='left')
    df_m['Aus_Total_real'] = df_m[COLUMNAS_AUSENCIA].sum(axis=1)
    pred_aus_cols = [f'{c}_pred' for c in COLUMNAS_AUSENCIA if f'{c}_pred' in df_m.columns]
    df_m['Aus_Total_pred'] = df_m[pred_aus_cols].sum(axis=1)
    dot_pred = 'Dotacion_pred' if 'Dotacion_pred' in df_m.columns else 'Dotacion'
    mask = (df_m['Dotacion'] > 0) & (df_m[dot_pred] > 0)
    df_m = df_m[mask]
    if len(df_m) == 0:
        print(f"  {label}: sin datos")
        return
    abs_r = df_m['Aus_Total_real'] / df_m['Dotacion'] * 100
    abs_p = df_m['Aus_Total_pred'] / df_m[dot_pred] * 100
    ms = calc_metrics_detailed(abs_r.values, abs_p.values)
    print(f"  {label:<12} MAPE={ms.get('MAPE',0):.2f}%  WMAPE={ms.get('WMAPE',0):.2f}%  Bias={ms.get('Bias(ME)',0):+.2f}pp  R2={ms.get('R2',0):.4f}")

_abs_total_insample(hist_preds,      'CatBoost')



DIAGNOSTICO DE PERFORMANCE - in-sample  (CatBoost · LightGBM · ElasticNet)

Calculando metricas in-sample...

Variable                 MAPE%   WMAPE%   SMAPE%    MAE%   Bias(ME)      R2  n_sitios
-------------------------------------------------------------------------------------
Dotacion                 60.29    10.18    17.31   10.18      293.9  0.7470        50
Falta_Inj               261.72    13.15    31.27   13.15       19.6  0.8605        50
Atestados_Medicos        73.85    13.92    23.22   13.92       13.3  0.7605        50
Rest_Abs_Gestionable     39.01    19.11    27.28   19.11        0.3  0.6664        50

SEMAFORO DE SALUD POR VARIABLE (CatBoost WMAPE)
  🔴 Dotacion               WMAPE=10.2%  Bias=+293.9 (sobreestima)
  🔴 Falta_Inj              WMAPE=13.2%  Bias=+19.6 (sobreestima)
  🔴 Atestados_Medicos      WMAPE=13.9%  Bias=+13.3 (sobreestima)
  🔴 Rest_Abs_Gestionable   WMAPE=19.1%  Bias=+0.3 (sobreestima)

PERFORMANCE ABS_TOTAL — CatBoost vs LightGBM vs ElasticNet (in-s

---
### Celda 6 — Forecast Futuro CatBoost por Componente (hasta Dic 2027)

In [6]:
# =============================================================================
# GENERAR FORECAST FUTURO — CatBoost · LightGBM · ElasticNet (por componente)
# =============================================================================

print("=" * 70)
print(f"GENERANDO FORECAST ({N_MONTHS_FORECAST} meses hasta {FECHA_FIN_FORECAST.strftime('%Y-%m')})")
print("=" * 70)

all_forecasts      = {}

for col in COLUMNAS_FORECAST:
    print(f"\nForecast: {col}...")
    all_forecasts[col]      = models[col].predict_future(FECHAS_FORECAST)
    print(f"   CatBoost={len(all_forecasts[col])} predicciones")

print("\nForecast por componente completado (3 modelos ML)")


GENERANDO FORECAST (18 meses hasta 2027-12)

Forecast: Dotacion...
   CatBoost=900 predicciones

Forecast: Falta_Inj...
   CatBoost=900 predicciones

Forecast: Atestados_Medicos...
   CatBoost=900 predicciones

Forecast: Rest_Abs_Gestionable...
   CatBoost=900 predicciones

Forecast por componente completado (3 modelos ML)


---
### Celda 7 — Validacion Walk-Forward CatBoost sobre ABS Total (Comparable con TSS)

In [7]:
# =============================================================================
# VALIDACION WALK-FORWARD — CatBoost · LightGBM · ElasticNet sobre ABS_TOTAL
# =============================================================================

print(f"Walk-forward validation (cutoff {VAL_CUTOFF.strftime('%Y-%m')}, horizon={VAL_HORIZON} meses)...")

df_train_val = df_data[df_data['fecha_mes'] <= VAL_CUTOFF].copy()
df_test_val  = df_data[
    (df_data['fecha_mes'] > VAL_CUTOFF) &
    (df_data['fecha_mes'] <= VAL_CUTOFF + relativedelta(months=VAL_HORIZON))
].copy()

meses_x_sitio_val = df_train_val.groupby('Operational_name')['fecha_mes'].count()
SITIOS_VAL      = sorted(meses_x_sitio_val[meses_x_sitio_val >= 6].index.tolist())
FECHAS_VAL_TEST = sorted(df_test_val['fecha_mes'].unique())

def _run_val_forecast(ForecasterClass, df_train, sites, fechas_test):
    # Entrena con datos hasta VAL_CUTOFF y devuelve ABS_Total predicho en el test.
    val_fcs = {}
    for col in COLUMNAS_FORECAST:
        m = ForecasterClass(target_col=col, name=f'{col}_val')
        m.fit(df_train, sites)
        val_fcs[col] = m.predict_future(fechas_test)
    wide = None
    for col in COLUMNAS_FORECAST:
        df_fc = val_fcs[col].rename(columns={f'{col}_forecast': col})
        if len(df_fc) == 0:
            continue
        if wide is None:
            wide = df_fc[['fecha_mes', 'Operational_name', col]].copy()
        else:
            wide = wide.merge(df_fc[['fecha_mes', 'Operational_name', col]], on=['fecha_mes', 'Operational_name'], how='outer')
    for col in COLUMNAS_FORECAST:
        if col not in wide.columns: wide[col] = 0
        wide[col] = wide[col].fillna(0)
    wide['Ausencia_Total_pred'] = wide[COLUMNAS_AUSENCIA].sum(axis=1)
    if 'Dotacion' in wide.columns:
        wide = wide.rename(columns={'Dotacion': 'Dotacion_pred'})
    return wide

def _build_val_df(wide, model_col_name):
    # Merge con df_test_val y calcula ABS_real + ABS_predicho.
    df_v = df_test_val[['fecha_mes', 'Operational_name', 'Ausencia_Total', 'Dotacion']].merge(
        wide[['fecha_mes', 'Operational_name', 'Ausencia_Total_pred'] +
             (['Dotacion_pred'] if 'Dotacion_pred' in wide.columns else [])],
        on=['fecha_mes', 'Operational_name'], how='inner')
    dot_pred = 'Dotacion_pred' if 'Dotacion_pred' in df_v.columns else 'Dotacion'
    df_v['ABS_real']     = np.where(df_v['Dotacion'] > 0, df_v['Ausencia_Total'] / df_v['Dotacion'] * 100, 0).round(4)
    df_v[model_col_name] = np.where(df_v[dot_pred] > 0,   df_v['Ausencia_Total_pred'] / df_v[dot_pred] * 100, 0).round(4)
    return df_v

wide_cb   = _run_val_forecast(CatBoostTTEForecaster,   df_train_val, SITIOS_VAL, FECHAS_VAL_TEST)

df_cb_val_base = _build_val_df(wide_cb,   'ABS_catboost_val')

df_cb_val = df_cb_val_base.copy()

for col_val, nombre in [('ABS_catboost_val', 'CatBoost')]:
    if col_val in df_cb_val.columns:
        mape = calcular_mape(df_cb_val['ABS_real'].values, df_cb_val[col_val].values)
        print(f"  {nombre:<12} validacion: {len(df_cb_val)} reg | MAPE={mape:.2f}%")


Walk-forward validation (cutoff 2025-12, horizon=6 meses)...

Entrenando CatBoostTTEForecaster para Dotacion_val...
   45 modelos entrenados de 54 sitios

Entrenando CatBoostTTEForecaster para Falta_Inj_val...
   45 modelos entrenados de 54 sitios

Entrenando CatBoostTTEForecaster para Atestados_Medicos_val...
   45 modelos entrenados de 54 sitios

Entrenando CatBoostTTEForecaster para Rest_Abs_Gestionable_val...
   45 modelos entrenados de 54 sitios
  CatBoost     validacion: 248 reg | MAPE=38.56%


---
### Celda 11 — Consolidacion CatBoost Agregado: ABS Total Historico + Forecast

In [8]:
# =============================================================================
# CONSOLIDAR AGREGADO ABS_TOTAL — CatBoost · LightGBM · ElasticNet
# =============================================================================

def _build_agg_df(hp_dict, fc_dict, abs_col_name):
    # Suma componentes por modelo para armar ABS_Total historico + forecast.
    dh = df_data[['fecha_mes', 'Operational_name', 'Dotacion', 'Ausencia_Total']].copy()
    for col in COLUMNAS_FORECAST:
        if col in hp_dict:
            dh = dh.merge(hp_dict[col], on=['fecha_mes', 'Operational_name'], how='left')
        else:
            dh[f'{col}_pred'] = np.nan
    for col in COLUMNAS_FORECAST:
        dh[f'{col}_pred'] = dh[f'{col}_pred'].fillna(0)
    dh['Ausencia_Total_pred'] = dh[[f'{c}_pred' for c in COLUMNAS_AUSENCIA]].sum(axis=1)
    dh['TIPO'] = 'Historical'
    dh['ABS_real'] = np.where(dh['Dotacion'] > 0, (dh['Ausencia_Total'] / dh['Dotacion']) * 100, 0).round(4)
    dot_pred = 'Dotacion_pred' if 'Dotacion_pred' in dh.columns else 'Dotacion'
    dh[abs_col_name] = np.where(dh[dot_pred] > 0, (dh['Ausencia_Total_pred'] / dh[dot_pred]) * 100, 0).round(4)

    df_fc_wide = None
    for col in COLUMNAS_FORECAST:
        df_fc = fc_dict[col].copy().rename(columns={f'{col}_forecast': col})
        if len(df_fc) == 0: continue
        if df_fc_wide is None:
            df_fc_wide = df_fc[['fecha_mes', 'Operational_name', col]].copy()
        else:
            df_fc_wide = df_fc_wide.merge(df_fc[['fecha_mes', 'Operational_name', col]], on=['fecha_mes', 'Operational_name'], how='outer')
    for col in COLUMNAS_FORECAST:
        if col not in df_fc_wide.columns: df_fc_wide[col] = 0
        df_fc_wide[col] = df_fc_wide[col].fillna(0)
    df_fc_wide['Ausencia_Total_pred'] = df_fc_wide[COLUMNAS_AUSENCIA].sum(axis=1)
    if 'Dotacion' in df_fc_wide.columns:
        df_fc_wide = df_fc_wide.rename(columns={'Dotacion': 'Dotacion_pred'})
    df_fc_wide['TIPO'] = 'Forecast'
    df_fc_wide['ABS_real'] = np.nan
    dot_pred_fc = 'Dotacion_pred' if 'Dotacion_pred' in df_fc_wide.columns else None
    if dot_pred_fc:
        df_fc_wide[abs_col_name] = np.where(df_fc_wide[dot_pred_fc] > 0, (df_fc_wide['Ausencia_Total_pred'] / df_fc_wide[dot_pred_fc]) * 100, 0).round(4)
    else:
        df_fc_wide[abs_col_name] = 0

    dh = dh.drop_duplicates(subset=['Operational_name', 'fecha_mes'], keep='last')
    df_fc_wide = df_fc_wide.drop_duplicates(subset=['Operational_name', 'fecha_mes'], keep='last')
    return pd.concat([
        dh[['Operational_name', 'fecha_mes', 'TIPO', 'ABS_real', abs_col_name]],
        df_fc_wide[['Operational_name', 'fecha_mes', 'TIPO', 'ABS_real', abs_col_name]]
    ], ignore_index=True)

df_cb_agg   = _build_agg_df(hist_preds,      all_forecasts,      'ABS_catboost').drop_duplicates(subset=['Operational_name', 'fecha_mes'], keep='last')

df_cb = df_cb_agg.copy()

print(f"Agregado ABS_Total: {len(df_cb):,} reg | {df_cb['Operational_name'].nunique()} sitios")



Agregado ABS_Total: 2,243 reg | 64 sitios


---
### Celda 14 — Consolidacion df_final: Historico + Forecast por Componente (con MAPE)

In [9]:
# =============================================================================
# CONSOLIDAR df_final POR MODELO: HISTORICO (real+pred) + FORECAST POR COMPONENTE
# =============================================================================
# Detalle por componente (Dotacion, Falta_Inj, Atestados_Medicos,
# Rest_Abs_Gestionable) para CatBoost, LightGBM y ElasticNet — usado por el
# dashboard (selector de modelo en la vista "Componentes") y por la
# exportacion a BigQuery (que usa el de CatBoost).

def build_df_final(hist_preds_dict, forecasts_dict):
    df_historico_out = df_data[['fecha_mes', 'Operational_name'] + COLUMNAS_FORECAST].copy()
    df_historico_out['tipo'] = 'historico'

    for col in COLUMNAS_FORECAST:
        if col in hist_preds_dict:
            df_historico_out = df_historico_out.merge(
                hist_preds_dict[col][['fecha_mes', 'Operational_name', f'{col}_pred']],
                on=['fecha_mes', 'Operational_name'], how='left'
            )
        else:
            df_historico_out[f'{col}_pred'] = np.nan

    df_forecast_combined = None
    for col in COLUMNAS_FORECAST:
        df_fc = forecasts_dict[col].copy()
        if len(df_fc) == 0:
            continue
        df_fc = df_fc.rename(columns={f'{col}_forecast': col})
        df_fc[f'{col}_pred'] = df_fc[col]
        if df_forecast_combined is None:
            df_forecast_combined = df_fc[['fecha_mes', 'Operational_name', col, f'{col}_pred']].copy()
        else:
            df_forecast_combined = df_forecast_combined.merge(
                df_fc[['fecha_mes', 'Operational_name', col, f'{col}_pred']],
                on=['fecha_mes', 'Operational_name'], how='outer'
            )

    if df_forecast_combined is not None:
        df_forecast_combined['tipo'] = 'forecast'
        df_out = pd.concat([df_historico_out, df_forecast_combined], ignore_index=True)
    else:
        df_out = df_historico_out.copy()

    for col in COLUMNAS_FORECAST:
        df_out[col] = df_out[col].fillna(0)
        df_out[f'{col}_pred'] = df_out[f'{col}_pred'].fillna(0)

    for col in COLUMNAS_AUSENCIA:
        df_out[f'ABS_{col}'] = np.where(df_out['Dotacion'] > 0, (df_out[col] / df_out['Dotacion']).round(4), 0)
        df_out[f'ABS_{col}_pred'] = np.where(df_out['Dotacion_pred'] > 0, (df_out[f'{col}_pred'] / df_out['Dotacion_pred']).round(4), 0)

    df_out['Ausencia_Total'] = df_out[COLUMNAS_AUSENCIA].sum(axis=1)
    df_out['Ausencia_Total_pred'] = df_out[[f'{c}_pred' for c in COLUMNAS_AUSENCIA]].sum(axis=1)
    df_out['ABS_Total'] = np.where(df_out['Dotacion'] > 0, (df_out['Ausencia_Total'] / df_out['Dotacion']).round(4), 0)
    df_out['ABS_Total_pred'] = np.where(df_out['Dotacion_pred'] > 0, (df_out['Ausencia_Total_pred'] / df_out['Dotacion_pred']).round(4), 0)

    mask_hist = df_out['tipo'] == 'historico'
    for col in COLUMNAS_FORECAST:
        mape_col = f'MAPE_{col}'
        df_out[mape_col] = np.nan
        m = mask_hist & (df_out[col] > 0) & (df_out[f'{col}_pred'] > 0)
        df_out.loc[m, mape_col] = (
            np.abs(df_out.loc[m, col] - df_out.loc[m, f'{col}_pred']) / df_out.loc[m, col] * 100
        ).round(2)

    mask_abs = mask_hist & (df_out['ABS_Total'] > 0)
    df_out['MAPE_ABS_Total'] = np.nan
    df_out.loc[mask_abs, 'MAPE_ABS_Total'] = (
        np.abs(df_out.loc[mask_abs, 'ABS_Total'] - df_out.loc[mask_abs, 'ABS_Total_pred']) / df_out.loc[mask_abs, 'ABS_Total'] * 100
    ).round(2)

    df_out['MAPE_STATUS'] = np.where(
        df_out['MAPE_ABS_Total'].isna(), '',
        np.where(df_out['MAPE_ABS_Total'] < 5, 'Alta Precision',
        np.where(df_out['MAPE_ABS_Total'] < 10, 'Desvio', 'Revisar')))

    df_out['ANO'] = df_out['fecha_mes'].dt.year
    df_out['MES'] = df_out['fecha_mes'].dt.month
    return df_out.sort_values(['Operational_name', 'fecha_mes']).reset_index(drop=True)


df_final_catboost = build_df_final(hist_preds, all_forecasts)
df_final_by_model = {'CatBoost': df_final_catboost}
df_final = df_final_catboost

n_hist = len(df_final[df_final['tipo'] == 'historico'])
n_fc   = len(df_final[df_final['tipo'] == 'forecast'])
print(f"df_final CatBoost: {len(df_final):,} reg (hist={n_hist:,}, forecast={n_fc:,}) | {df_final['Operational_name'].nunique()} sitios")


df_final CatBoost: 3,174 reg (hist=2,274, forecast=900) | 64 sitios


---
### Celda 15 — Agregacion por Macro Region (sitios + macro regiones en un solo CSV)

In [10]:
# =============================================================================
# CELDA 15 — Agregacion por Macro Region
# Join correcto: UPPER(SITE de la tabla BQ) = UPPER(Site del agrupador)
# =============================================================================

AGRUP_PATH = os.path.join(os.path.dirname(OUTPUT_DIR), 'Agrupadores transportes new.csv')

df_macro_out = pd.DataFrame()  # default vacio si falla

try:
    agrup = pd.read_csv(AGRUP_PATH, encoding='latin-1', sep=None, engine='python')
    col_site  = agrup.columns[1]   # 'Site'
    col_macro = agrup.columns[4]   # 'Macro region'
    agrup[col_macro] = agrup[col_macro].str.strip().replace({'rest of SP': 'Rest of SP'})

    # Mapeo UPPER(Site) -> Macro_Region
    site_to_macro = {str(s).upper().strip(): m
                     for s, m in zip(agrup[col_site], agrup[col_macro])
                     if pd.notna(s) and pd.notna(m)}

    print('Agrupadores: ' + str(len(site_to_macro)) + ' sitios | Macros: ' + str(sorted(set(site_to_macro.values()))))

    # Crear mapeo Operational_name -> Macro_Region usando SITE de df_data
    # df_data tiene SITE (campo original de BQ) y Operational_name
    op_site_map = df_data[['Operational_name','SITE']].drop_duplicates().copy()
    op_site_map['SITE_upper'] = op_site_map['SITE'].astype(str).str.upper().str.strip()
    op_site_map['Macro_Region'] = op_site_map['SITE_upper'].map(site_to_macro)

    op_to_macro = op_site_map.dropna(subset=['Macro_Region']).set_index('Operational_name')['Macro_Region'].to_dict()
    matched_n = len(op_to_macro)
    total_n   = df_data['Operational_name'].nunique()
    print('Match via SITE (UPPER): ' + str(matched_n) + '/' + str(total_n) + ' sitios')

    # Sitios sin match
    no_match = [op for op in df_data['Operational_name'].unique() if op not in op_to_macro]
    if no_match:
        print('Sin macro region (' + str(len(no_match)) + '): ' + str(no_match[:5]))

    COMP_AUSENCIA = ['Falta_Inj', 'Atestados_Medicos', 'Rest_Abs_Gestionable']

    df_m = df_final.copy()
    df_m['fecha_mes'] = pd.to_datetime(df_m['fecha_mes'])
    df_m['_macro'] = df_m['Operational_name'].map(op_to_macro)
    matched = df_m[df_m['_macro'].notna()].copy()

    num_cols = ['Dotacion','Dotacion_pred'] + COMP_AUSENCIA + [c+'_pred' for c in COMP_AUSENCIA]
    num_cols = [c for c in num_cols if c in matched.columns]
    agg_macro = matched.groupby(['_macro','fecha_mes','tipo'])[num_cols].sum().reset_index()

    rows_macro = []
    for _, r in agg_macro.iterrows():
        tipo = r['tipo']
        dot_r = max(r.get('Dotacion', 0), 0)
        dot_p = max(r.get('Dotacion_pred', 0), dot_r)
        row = {
            'Operational_name': r['_macro'],
            'fecha_mes': r['fecha_mes'].strftime('%Y-%m-%d'),
            'TYPE': 'Historico' if tipo == 'historico' else 'Forecast',
        }
        if tipo == 'historico':
            aus_r = sum(r.get(c, 0) for c in COMP_AUSENCIA)
            row['ABS_Total_Real']    = round(aus_r / dot_r, 4) if dot_r > 0 else 0
            row['ABS_Falta_Real']    = round(r.get('Falta_Inj', 0) / dot_r, 4) if dot_r > 0 else 0
            row['ABS_Atestado_Real'] = round(r.get('Atestados_Medicos', 0) / dot_r, 4) if dot_r > 0 else 0
            row['ABS_Rest_Real']     = round(r.get('Rest_Abs_Gestionable', 0) / dot_r, 4) if dot_r > 0 else 0
        else:
            row['ABS_Total_Real']    = float('nan')
            row['ABS_Falta_Real']    = float('nan')
            row['ABS_Atestado_Real'] = float('nan')
            row['ABS_Rest_Real']     = float('nan')
        aus_p = sum(r.get(c+'_pred', 0) for c in COMP_AUSENCIA)
        row['ABS_Total_CatBoost']    = round(aus_p / dot_p, 4) if dot_p > 0 else 0
        row['ABS_Falta_CatBoost']    = round(r.get('Falta_Inj_pred', 0) / dot_p, 4) if dot_p > 0 else 0
        row['ABS_Atestado_CatBoost'] = round(r.get('Atestados_Medicos_pred', 0) / dot_p, 4) if dot_p > 0 else 0
        row['ABS_Rest_CatBoost']     = round(r.get('Rest_Abs_Gestionable_pred', 0) / dot_p, 4) if dot_p > 0 else 0
        rows_macro.append(row)

    df_macro_out = pd.DataFrame(rows_macro).sort_values(['Operational_name','fecha_mes']).reset_index(drop=True)
    print('Macro regiones generadas: ' + str(df_macro_out['Operational_name'].nunique()))
    print(str(sorted(df_macro_out['Operational_name'].unique())))

except FileNotFoundError:
    print('AVISO: No se encontro: ' + AGRUP_PATH)
except Exception as e:
    print('ERROR macro region: ' + str(e))
    import traceback; traceback.print_exc()


Agrupadores: 196 sitios | Macros: ['Cajamar', 'Centro/sureste', 'Minas', 'Norte/noreste', 'Rest of SP', 'Sur']
Match via SITE (UPPER): 45/64 sitios
Sin macro region (19): ['RC - SUMARE RC02', 'SC - ARACATUBA SSP10', 'SC - BIGUACU SSC2', 'SC - FLORIANOPOLIS', 'SC - GUARULHOS SSP19']
Macro regiones generadas: 4
['Cajamar', 'Minas', 'Rest of SP', 'Sur']


In [11]:
# =============================================================================
# CELDA 16 — GUARDAR CSV FINAL UNICO
# Sitios individuales + macro regiones en un solo archivo.
# Formato: sep=; decimal=, | valores en ratio (0,083 = 8,3% ABS)
# =============================================================================

print('=' * 70)
print('Guardando CSVs en: ' + OUTPUT_DIR)
print('=' * 70)

def save_df(df, fname):
    df2 = df.copy()
    if 'fecha_mes' in df2.columns:
        df2['fecha_mes'] = pd.to_datetime(df2['fecha_mes']).dt.strftime('%Y-%m-%d')
    fpath = os.path.join(OUTPUT_DIR, fname)
    df2.to_csv(fpath, index=False, sep=';', decimal=',', encoding='utf-8-sig')
    print('  ' + fname + ' (' + str(len(df2)) + ' filas)')

# CSVs intermedios de soporte
save_df(df_final,  'catboost_ABS_TTE_componentes.csv')
save_df(df_cb,     'catboost_ABS_TTE_produccion_agregado.csv')
if len(df_cb_val) > 0:
    save_df(df_cb_val, 'catboost_ABS_TTE_validacion.csv')

# =============================================================================
# CSV FINAL UNICO: sitios individuales + macro regiones
# =============================================================================
print()
print('Generando CSV FINAL UNICO...')

LABEL_MAP = {'Falta_Inj': 'Falta', 'Atestados_Medicos': 'Atestado', 'Rest_Abs_Gestionable': 'Rest_Gest'}
COMP_ABS  = ['Falta_Inj', 'Atestados_Medicos', 'Rest_Abs_Gestionable']

def agg_model(df):
    df2 = df.copy()
    df2['fecha_mes'] = pd.to_datetime(df2['fecha_mes']).dt.strftime('%Y-%m-%d')
    num_cols = df2.select_dtypes(include='number').columns.tolist()
    return df2.groupby(['Operational_name', 'fecha_mes', 'tipo'], as_index=False)[num_cols].sum()

cb_agg = agg_model(df_final_by_model['CatBoost'])

# Sitios individuales (todos los 156)
out = cb_agg[['Operational_name', 'fecha_mes', 'tipo']].copy()
out['TYPE'] = cb_agg['tipo'].map({'historico': 'Historico', 'forecast': 'Forecast'})
mask_h = cb_agg['tipo'] == 'historico'
out['ABS_Total_Real']    = np.where(mask_h, cb_agg['ABS_Total'].round(4),                np.nan)
out['ABS_Falta_Real']    = np.where(mask_h, cb_agg['ABS_Falta_Inj'].round(4),            np.nan)
out['ABS_Atestado_Real'] = np.where(mask_h, cb_agg['ABS_Atestados_Medicos'].round(4),    np.nan)
out['ABS_Rest_Real']     = np.where(mask_h, cb_agg['ABS_Rest_Abs_Gestionable'].round(4), np.nan)
out['ABS_Total_CatBoost']    = cb_agg['ABS_Total_pred'].round(4)
out['ABS_Falta_CatBoost']    = cb_agg['ABS_Falta_Inj_pred'].round(4)
out['ABS_Atestado_CatBoost'] = cb_agg['ABS_Atestados_Medicos_pred'].round(4)
out['ABS_Rest_CatBoost']     = cb_agg['ABS_Rest_Abs_Gestionable_pred'].round(4)

df_sitios_final = out.drop(columns=['tipo']).sort_values(['Operational_name','fecha_mes']).reset_index(drop=True)

# Combinar sitios + macro regiones en un solo DataFrame
if len(df_macro_out) > 0:
    # Alinear columnas
    macro_cols = df_sitios_final.columns.tolist()
    df_macro_aligned = df_macro_out.reindex(columns=macro_cols)
    df_final_unico = pd.concat([df_sitios_final, df_macro_aligned], ignore_index=True)
    print('  Sitios individuales : ' + str(df_sitios_final['Operational_name'].nunique()))
    print('  Macro regiones      : ' + str(df_macro_out['Operational_name'].nunique()))
else:
    df_final_unico = df_sitios_final.copy()
    print('  Solo sitios (macro region no disponible)')

out_path = os.path.join(OUTPUT_DIR, 'ABS_TTE_CatBoost_FORECAST_sheets.csv')
df_final_unico.to_csv(out_path, index=False, sep=';', decimal=',', encoding='utf-8-sig')

print()
print('CSV FINAL: ABS_TTE_CatBoost_FORECAST_sheets.csv')
print('  Filas totales : ' + str(len(df_final_unico)))
print('  Meses forecast: ' + str(df_final_unico[df_final_unico['TYPE']=='Forecast']['fecha_mes'].nunique()))


Guardando CSVs en: c:\Users\jhocontreras\Desktop\TTE_TIME_SERIES\ABS GEST FINANCIAL\forecast_results_ABS_TTE_catboost
  catboost_ABS_TTE_componentes.csv (3174 filas)
  catboost_ABS_TTE_produccion_agregado.csv (2243 filas)
  catboost_ABS_TTE_validacion.csv (248 filas)

Generando CSV FINAL UNICO...
  Sitios individuales : 64
  Macro regiones      : 4

CSV FINAL: ABS_TTE_CatBoost_FORECAST_sheets.csv
  Filas totales : 2423
  Meses forecast: 18


---
### Celda 14b — Dashboard Interactivo ABS TTE (CatBoost Real vs Forecast)

In [12]:
# =============================================================================
# DASHBOARD INTERACTIVO ABS TTE — CatBoost (Real vs Forecast por componente)
# =============================================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import plotly.express as px

if 'df_final' not in dir() or len(df_final) == 0:
    print("ERROR: df_final no disponible. Ejecuta las celdas anteriores.")
else:
    sitios_disponibles = sorted(df_final['Operational_name'].unique().tolist())

    sitio_w  = widgets.Dropdown(options=sitios_disponibles, description='Sitio:',
                                layout=widgets.Layout(width='320px'))
    prev_w   = widgets.Button(description='◀', layout=widgets.Layout(width='40px'))
    next_w   = widgets.Button(description='▶', layout=widgets.Layout(width='40px'))
    output_w = widgets.Output()

    def nav(delta):
        opts = list(sitio_w.options)
        idx = opts.index(sitio_w.value) if sitio_w.value in opts else 0
        sitio_w.value = opts[(idx + delta) % len(opts)]

    def graficar(change=None):
        with output_w:
            clear_output(wait=True)
            site = sitio_w.value
            ds = df_final[df_final['Operational_name'] == site].sort_values('fecha_mes')
            dh = ds[ds['tipo'] == 'historico']
            dfc = ds[ds['tipo'] == 'forecast']
            hoy = pd.Timestamp.now().replace(day=1)
            comp_colors = px.colors.qualitative.T10

            # ── Grafico 1: ABS Total Real vs CatBoost ─────────────────────────
            fig1 = go.Figure()
            if len(dh) > 0:
                fig1.add_trace(go.Scatter(
                    x=dh['fecha_mes'], y=(dh['ABS_Total'] * 100).round(2),
                    mode='lines+markers', name='Real',
                    line=dict(color='#2ecc71', width=3), marker=dict(size=7),
                    hovertemplate='<b>Real</b><br>%{x|%Y-%m}: %{y:.2f}%<extra></extra>'
                ))
                if 'ABS_Total_pred' in dh.columns:
                    fig1.add_trace(go.Scatter(
                        x=dh['fecha_mes'], y=(dh['ABS_Total_pred'] * 100).round(2),
                        mode='lines+markers', name='CatBoost (in-sample)',
                        line=dict(color='#e74c3c', width=2, dash='dash'), marker=dict(size=5, symbol='square'),
                        hovertemplate='<b>CatBoost</b><br>%{x|%Y-%m}: %{y:.2f}%<extra></extra>'
                    ))
            if len(dfc) > 0:
                if len(dh) > 0 and 'ABS_Total_pred' in dh.columns:
                    fig1.add_trace(go.Scatter(
                        x=[dh['fecha_mes'].iloc[-1], dfc['fecha_mes'].iloc[0]],
                        y=[(dh['ABS_Total_pred'].iloc[-1]*100).round(2), (dfc['ABS_Total_pred'].iloc[0]*100).round(2)],
                        mode='lines', line=dict(color='#e74c3c', dash='dash', width=1),
                        showlegend=False, opacity=0.5
                    ))
                fig1.add_trace(go.Scatter(
                    x=dfc['fecha_mes'], y=(dfc['ABS_Total_pred'] * 100).round(2),
                    mode='lines+markers', name='CatBoost (forecast)',
                    line=dict(color='#e74c3c', width=2.5, dash='dash'), marker=dict(size=6, symbol='square'),
                    hovertemplate='<b>CatBoost Fc</b><br>%{x|%Y-%m}: %{y:.2f}%<extra></extra>'
                ))

            if len(dfc) > 0:
                fig1.add_vrect(x0=str(dfc['fecha_mes'].min()), x1=str(dfc['fecha_mes'].max()),
                               fillcolor='blue', opacity=0.04, line_width=0)
                fig1.add_annotation(x=str(dfc['fecha_mes'].min()), y=1.02, yref='paper',
                                    text='<b>Forecast</b>', showarrow=False,
                                    font=dict(color='blue', size=10), xanchor='left')
            fig1.add_vline(x=str(hoy), line_dash='dash', line_color='red', line_width=1.5)
            fig1.update_layout(
                title=dict(text=f'<b>{site}</b> — ABS Total: Real vs CatBoost', x=0.5, font_size=14),
                xaxis=dict(title='Fecha', tickformat='%b %Y', dtick='M3', tickangle=-45),
                yaxis=dict(title='ABS Total (%)'),
                hovermode='x unified', height=480, template='plotly_white',
                legend=dict(orientation='h', y=-0.25)
            )
            fig1.show()

            # ── Grafico 2: ABS por componente ─────────────────────────────────
            fig2 = go.Figure()
            for idx, comp in enumerate(COLUMNAS_AUSENCIA):
                col_r = f'ABS_{comp}'
                col_p = f'ABS_{comp}_pred'
                color = comp_colors[idx % len(comp_colors)]
                label = comp.replace('_Inj','').replace('Atestados_Medicos','Atestado').replace('Rest_Abs_Gestionable','Rest')
                if len(dh) > 0 and col_r in dh.columns:
                    fig2.add_trace(go.Scatter(
                        x=dh['fecha_mes'], y=(dh[col_r] * 100).round(2),
                        mode='lines+markers', name=label,
                        line=dict(color=color, width=2), marker=dict(size=5),
                        legendgroup=comp
                    ))
                if len(dfc) > 0 and col_p in dfc.columns:
                    if len(dh) > 0 and col_r in dh.columns:
                        fig2.add_trace(go.Scatter(
                            x=[dh['fecha_mes'].iloc[-1], dfc['fecha_mes'].iloc[0]],
                            y=[(dh[col_r].iloc[-1]*100).round(2), (dfc[col_p].iloc[0]*100).round(2)],
                            mode='lines', line=dict(color=color, dash='dash', width=1),
                            showlegend=False, legendgroup=comp, opacity=0.5
                        ))
                    fig2.add_trace(go.Scatter(
                        x=dfc['fecha_mes'], y=(dfc[col_p] * 100).round(2),
                        mode='lines+markers', name=f'{label} (Fc)',
                        line=dict(color=color, width=2, dash='dash'), marker=dict(size=4, symbol='square'),
                        showlegend=False, legendgroup=comp
                    ))

            if len(dfc) > 0:
                fig2.add_vrect(x0=str(dfc['fecha_mes'].min()), x1=str(dfc['fecha_mes'].max()),
                               fillcolor='blue', opacity=0.04, line_width=0)
            fig2.add_vline(x=str(hoy), line_dash='dash', line_color='red', line_width=1.5)
            fig2.update_layout(
                title=f'{site} — ABS por Componente (historico solido | forecast punteado)',
                xaxis=dict(title='Fecha', tickformat='%b %Y', dtick='M3', tickangle=-45),
                yaxis=dict(title='ABS (%)'),
                hovermode='x unified', height=420, template='plotly_white',
                legend=dict(orientation='h', y=-0.3)
            )
            fig2.show()

            # ── Grafico 3: Dotacion ────────────────────────────────────────────
            fig3 = go.Figure()
            if len(dh) > 0 and 'Dotacion' in dh.columns:
                fig3.add_trace(go.Scatter(x=dh['fecha_mes'], y=dh['Dotacion'],
                                          mode='lines+markers', name='Dotacion Real',
                                          line=dict(color='#2c3e50', width=2.5), marker=dict(size=6)))
            if len(dfc) > 0 and 'Dotacion_pred' in dfc.columns:
                if len(dh) > 0:
                    fig3.add_trace(go.Scatter(
                        x=[dh['fecha_mes'].iloc[-1], dfc['fecha_mes'].iloc[0]],
                        y=[dh['Dotacion'].iloc[-1], dfc['Dotacion_pred'].iloc[0]],
                        mode='lines', line=dict(color='#2c3e50', dash='dash', width=1),
                        showlegend=False, opacity=0.5
                    ))
                fig3.add_trace(go.Scatter(x=dfc['fecha_mes'], y=dfc['Dotacion_pred'],
                                          mode='lines+markers', name='Dotacion Forecast',
                                          line=dict(color='#2c3e50', width=2, dash='dash'),
                                          marker=dict(size=5, symbol='square')))
            fig3.add_vline(x=str(hoy), line_dash='dash', line_color='red', line_width=1.5)
            fig3.update_layout(title=f'{site} — Dotacion Real vs Forecast',
                               xaxis_title='Fecha', yaxis_title='Dotacion',
                               height=350, template='plotly_white', hovermode='x unified')
            fig3.show()

            # ── MAPE in-sample ─────────────────────────────────────────────────
            if 'MAPE_ABS_Total' in dh.columns:
                dm = dh[dh['MAPE_ABS_Total'].notna()]
                if len(dm) > 0:
                    mape_avg = dm['MAPE_ABS_Total'].mean()
                    n_ok  = (dm['MAPE_ABS_Total'] < 5).sum()
                    n_mid = ((dm['MAPE_ABS_Total'] >= 5) & (dm['MAPE_ABS_Total'] < 10)).sum()
                    n_bad = (dm['MAPE_ABS_Total'] >= 10).sum()
                    print(f'MAPE in-sample ABS_Total: {mape_avg:.2f}% | '
                          f'Alta precision (<5%): {n_ok} | Desvio (5-10%): {n_mid} | Revisar (>10%): {n_bad}')

    prev_w.on_click(lambda b: nav(-1))
    next_w.on_click(lambda b: nav(+1))
    sitio_w.observe(graficar, names='value')

    display(widgets.VBox([
        widgets.HBox([prev_w, sitio_w, next_w]),
        output_w
    ]))
    print('Dashboard ABS TTE CatBoost — navegá con las flechas o seleccioná el sitio.')
    graficar()


Dashboard ABS TTE CatBoost — navegá con las flechas o seleccioná el sitio.
